In [1]:
import sys
print(sys.executable)

/Users/medhapaul/arrhythmia-classification/.venv/bin/python


In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import sys

# Make the repository root importable
ROOT = Path.cwd()

if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import (
    RANDOM_STATE,
    N_SPLITS,
    DATA_PATH,
    CLEAN_PATH,
    IMAGE_DIR,
    DROP_CLASS,
    NORMAL_CLASS,
)

raw = pd.read_csv(
    ROOT / DATA_PATH,
    header=None,
    na_values="?"
)

print("Original dataset shape:", raw.shape)
print("Number of columns:", len(raw.columns))
print("Missing values:", raw.isna().sum().sum())

assert raw.shape == (452, 280), (
    f"Unexpected dataset shape: {raw.shape}"
)

display(raw.head())


Original dataset shape: (452, 280)
Number of columns: 280
Missing values: 408


,0,1,2,3,4,5,6,7,8,9,...,270,271,272,273,274,275,276,277,278,279
0,75,0,190,80,91,193,371,174,121,-16,...,0.0,9.0,-0.9,0.0,0.0,0.9,2.9,23.3,49.4,8
1,56,1,165,64,81,174,401,149,39,25,...,0.0,8.5,0.0,0.0,0.0,0.2,2.1,20.4,38.8,6
2,54,0,172,95,138,163,386,185,102,96,...,0.0,9.5,-2.4,0.0,0.0,0.3,3.4,12.3,49.0,10
3,55,0,175,94,100,202,380,179,143,28,...,0.0,12.2,-2.2,0.0,0.0,0.4,2.6,34.6,61.6,1
4,75,0,190,80,88,181,360,177,103,-16,...,0.0,13.1,-3.6,0.0,0.0,-0.1,3.9,25.4,62.8,7


In [4]:

TARGET_COL = raw.columns[-1]

print("Target column:", TARGET_COL)
print("\nOriginal class distribution:")

display(
    raw[TARGET_COL]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMissing values in target:", raw[TARGET_COL].isna().sum())


Target column: 279

Original class distribution:


279
1     245
2      44
3      15
4      15
5      13
6      25
7       3
8       2
9       9
10     50
14      4
15      5
16     22
Name: count, dtype: int64


Missing values in target: 0


In [5]:
before_drop = len(raw)

df = raw.loc[raw[TARGET_COL] != DROP_CLASS].copy()

removed_rows = before_drop - len(df)

print("Rows before:", before_drop)
print("Class 16 rows removed:", removed_rows)
print("Rows after:", len(df))

assert removed_rows == 22
assert len(df) == 430


Rows before: 452
Class 16 rows removed: 22
Rows after: 430


In [6]:
df[TARGET_COL] = np.where(
    df[TARGET_COL] == NORMAL_CLASS,
    0,
    1
).astype(int)

print("Binary class distribution:")
display(df[TARGET_COL].value_counts().sort_index())

print("\nClass proportions:")
display(df[TARGET_COL].value_counts(normalize=True).sort_index())

assert df[TARGET_COL].value_counts().to_dict() == {
    0: 245,
    1: 185
}


Binary class distribution:


279
0    245
1    185
Name: count, dtype: int64


Class proportions:


279
0    0.569767
1    0.430233
Name: proportion, dtype: float64

HANDLE MISSING VALUES

In [7]:
missing_counts = (
    df.drop(columns=[TARGET_COL])
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Columns with the most missing values:")
display(missing_counts.head(15))

print("\nColumns with any missing values:", (missing_counts > 0).sum())


Columns with the most missing values:


13     358
11      19
10       7
12       1
14       1
191      0
190      0
189      0
188      0
187      0
186      0
0        0
193      0
185      0
184      0
dtype: int64


Columns with any missing values: 5


In [8]:
# Replace with the verified integer index of the J feature.
J_COL = 13

assert J_COL in df.columns
assert J_COL != TARGET_COL

print("J feature index:", J_COL)
print("Missing values in J:", df[J_COL].isna().sum())
print("Percentage missing:", round(df[J_COL].isna().mean() * 100, 2), "%")

df = df.drop(columns=[J_COL])

print("Shape after dropping J:", df.shape)


J feature index: 13
Missing values in J: 358
Percentage missing: 83.26 %
Shape after dropping J: (430, 279)


REMOVE ZERO VARIANCE FEATURES

In [9]:

feature_cols = [c for c in df.columns if c != TARGET_COL]

nunique = df[feature_cols].nunique(dropna=True)

zero_variance_cols = nunique[nunique <= 1].index.tolist()

print("Zero-variance columns:", zero_variance_cols)
print("Number of zero-variance columns:", len(zero_variance_cols))

df = df.drop(columns=zero_variance_cols)

print("Shape after removing zero-variance columns:", df.shape)


Zero-variance columns: [19, 67, 69, 83, 131, 132, 139, 141, 143, 145, 151, 156, 157, 164, 204, 264, 274]
Number of zero-variance columns: 17
Shape after removing zero-variance columns: (430, 262)


Check height, weight, age for impossible values

In [10]:

for col, name in [(0, "Age"), (2, "Height"), (3, "Weight")]:
    if col not in df.columns:
        print(f"{name}: column {col} was removed; verify preprocessing.")
        continue

    values = pd.to_numeric(df[col], errors="coerce")

    print(f"\n{name} (column {col})")
    print("Minimum:", values.min())
    print("Maximum:", values.max())
    print("Missing:", values.isna().sum())

    display(
        df.loc[
            values.notna() & ((values <= 0) | (values > 120)),
            [col, TARGET_COL]
        ].head(20)
    )



Age (column 0)
Minimum: 0
Maximum: 83
Missing: 0


,0,279
316,0,1



Height (column 2)
Minimum: 105
Maximum: 780
Missing: 0


,2,279
0,190,1
1,165,1
2,172,1
3,175,0
4,190,1
5,169,1
6,160,0
7,162,0
8,168,0
9,167,1



Weight (column 3)
Minimum: 6
Maximum: 176
Missing: 0


,3,279
213,176,1
254,124,0


In [11]:

# Inspect unusual demographic measurements.
# Row indices shown here are DataFrame indices, not necessarily patient IDs.

print("Age = 0")
display(df.loc[df[0] == 0, [0, 2, 3, 279]])

print("Height > 300")
display(df.loc[df[2] > 300, [0, 2, 3, 279]].sort_values(2))

print("Weight < 30")
display(df.loc[df[3] < 30, [0, 2, 3, 279]].sort_values(3))


Age = 0


,0,2,3,279
316,0,608,10,1


Height > 300


,0,2,3,279
316,0,608,10,1
141,1,780,6,1


Weight < 30


,0,2,3,279
141,1,780,6,1
60,1,110,10,1
316,0,608,10,1
320,3,105,12,1
126,58,186,18,0
424,7,119,21,1
403,7,127,22,1
210,11,124,25,1
401,9,120,25,1
379,8,120,28,1


In [12]:

print("All distinct height values above 250:")
display(
    df.loc[df[2] > 250, [2]]
      .value_counts()
      .sort_index()
)

print("Number of heights above 250:", (df[2] > 250).sum())
print("Number of heights above 300:", (df[2] > 300).sum())


All distinct height values above 250:


2  
608    1
780    1
Name: count, dtype: int64

Number of heights above 250: 2
Number of heights above 300: 2


In [13]:

print("Records with age zero:")
display(df.loc[df[0] == 0, [0, 2, 3, 279]])

print("Records with weight below 30 kg:")
display(
    df.loc[df[3] < 30, [0, 2, 3, 279]]
      .sort_values(3)
)


Records with age zero:


,0,2,3,279
316,0,608,10,1


Records with weight below 30 kg:


,0,2,3,279
141,1,780,6,1
60,1,110,10,1
316,0,608,10,1
320,3,105,12,1
126,58,186,18,0
424,7,119,21,1
403,7,127,22,1
210,11,124,25,1
401,9,120,25,1
379,8,120,28,1


Demographic Data Quality Checks

Age, height and weight were inspected for missing, implausible and extreme values. Age ranged from 0 to 83 years, height ranged from 105 to 780 in the recorded units, and weight ranged from 6 to 176 kg.

Age zero, unusually large height values and very low weights were flagged for further investigation. These values were not automatically corrected or used as grounds for deleting records because their intended values could not be established from the measurements alone.

ID	Age	Height (recorded)	Weight (recorded)	Initial assessment
316	0	608	10	Multiple suspicious demographic values
141	1	780	6	Multiple suspicious demographic values
60	1	110	10	Very low weight; verify
320	3	105	12	Unusual, but don't assume erroneous
126	58	186	18	Very low weight for the recorded age
424	7	119	21	Needs verification

Plan of action: 
Height values of 608 and 780 are set to missing because they're implausible in centimetres.
Weight 18 at age 58 is set to missing, while low weights for children are retained for now.
Age 0 is retained because the record alone does not establish whether it is erroneous.
No rows are deleted. Your dataset should still contain 430 records, with 245 Normal and 185 Arrhythmia cases.

In [14]:

# Work on a copy so the earlier cleaned data remains available.
df = df.copy()

# Rule 1: Heights above 250 cm are treated as implausible.
# This is a screening threshold for our project, not an official UCI rule.
invalid_height = df[2] > 250

print("Implausible heights found:")
display(df.loc[invalid_height, [0, 2, 3, 279]])

df.loc[invalid_height, 2] = np.nan

# Rule 2: Flag extremely low weights for adults (age >= 18).
# Low weights for children are not automatically changed.
invalid_adult_weight = (df[0] >= 18) & (df[3] < 20)

print("Implausibly low adult weights found:")
display(df.loc[invalid_adult_weight, [0, 2, 3, 279]])

df.loc[invalid_adult_weight, 3] = np.nan

# Age zero is retained because its meaning cannot be established
# from this record alone.
print("Age-zero records retained:")
display(df.loc[df[0] == 0, [0, 2, 3, 279]])

print("\nRemaining missing values in demographic features:")
display(df[[0, 2, 3]].isna().sum())


Implausible heights found:


,0,2,3,279
141,1,780,6,1
316,0,608,10,1


Implausibly low adult weights found:


,0,2,3,279
126,58,186.0,18,0


Age-zero records retained:


,0,2,3,279
316,0,NaN,10.0,1



Remaining missing values in demographic features:


0    0
2    2
3    1
dtype: int64

In [16]:
print("Final shape:", df.shape)
print("\nBinary class counts:")
display(df[TARGET_COL].value_counts().sort_index())

print("\nMissing values by column:")
display(
    df.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
)

print("\nDuplicate rows:", df.duplicated().sum())

assert len(df) == 430
assert set(df[TARGET_COL].unique()) == {0, 1}
assert df[TARGET_COL].value_counts().to_dict() == {
    0: 245,
    1: 185
}


Final shape: (430, 262)

Binary class counts:


279
0    245
1    185
Name: count, dtype: int64


Missing values by column:


11    19
10     7
2      2
3      1
12     1
14     1
dtype: int64


Duplicate rows: 0


In [17]:
# Confirm the final cleaned dataset
print("Final shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values by column:")
display(df.isna().sum().loc[lambda s: s > 0])

print("\nTarget counts:")
display(df[279].value_counts().sort_index())

assert df.shape == (430, 262)
assert df[279].value_counts().to_dict() == {0: 245, 1: 185}
assert df[279].isna().sum() == 0

Final shape: (430, 262)
Total missing values: 31
Duplicate rows: 0

Missing values by column:


2      2
3      1
10     7
11    19
12     1
14     1
dtype: int64


Target counts:


279
0    245
1    185
Name: count, dtype: int64

In [18]:
clean_path = ROOT / CLEAN_PATH
df.to_csv(clean_path, index=False)

print("Saved to:", clean_path)
print("Saved shape:", pd.read_csv(clean_path).shape)

Saved to: /Users/medhapaul/arrhythmia-classification/Data/clean_data.csv
Saved shape: (430, 262)


## Preprocessing Summary

### 1. Dataset Loading
The UCI Arrhythmia dataset was loaded using Pandas, with `?` treated as missing values. The original dataset contained **452 records and 280 columns**, including the target class.

### 2. Class Filtering and Binary Label Conversion
The 22 records belonging to original class 16 were removed, leaving **430 records**. The remaining classes were converted into binary labels:
- **0 — Normal:** Original class 1 (245 records).
- **1 — Arrhythmia:** Original classes 2–15 (185 records).

The final binary class distribution was verified after preprocessing.

### 3. Missing-Value Analysis
Missing values were examined across the feature columns. The J feature, which had extensive missingness, was removed. Other missing feature values were retained for subsequent median imputation within the machine learning pipelines to avoid data leakage during cross-validation.

### 4. Demographic Data Quality Checks
Age, height and weight were inspected for implausible values. Age ranged from 0 to 83 years, recorded height ranged from 105 to 780 cm, and recorded weight ranged from 6 to 176 kg.

Two height values (608 and 780 cm) were treated as missing because they were implausible in the documented units. The weight value of 18 kg for a record with age 58 was also treated as missing. Other suspicious demographic values, including age zero and low weights among younger records, were retained because their intended values could not be established confidently.

No additional records were removed during these checks.

### 5. Feature Selection
Zero-variance features were identified and removed because they provided no variation across the dataset. The final dataset contained **430 records and 262 columns**, including the binary target column.

### 6. Final Dataset Validation
The cleaned dataset was checked for class counts, missing values and duplicate rows. The final class distribution remained 245 Normal records and 185 Arrhythmia records. A total of 31 missing feature values remained for handling within the modelling pipelines.

### 7. Output
The cleaned dataset was saved as `Data/clean_data.csv` and will be used by both team members for exploratory data analysis and model development.

### 8. Methodological Considerations
Stratified 5-fold cross-validation with `random_state=42` will be used for model evaluation. Imputation, feature scaling and PCA (where applicable) will be fitted within the cross-validation pipeline to prevent information leakage from validation folds into training.
